In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE

import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "aljarah/xAPI-Edu-Data",
  "xAPI-Edu-Data.csv",
)

In [3]:
df['at_risk'] = (df['Class'] == 'L').astype(int)
df['at_risk'].value_counts()

at_risk
0    353
1    127
Name: count, dtype: int64

In [4]:
df['engagement_score'] = df['raisedhands'] + df['VisITedResources'] + df['AnnouncementsView'] + df['Discussion']
df['is_absent_frequently'] = (df['StudentAbsenceDays'] == 'Above-7').astype(int)
df['parent_engaged'] = (df['ParentAnsweringSurvey'] == 'Yes').astype(int)
df['parent_satisfied'] = (df['ParentschoolSatisfaction'] == 'Good').astype(int)
df['is_male'] = (df['gender'] == 'M').astype(int)

In [5]:
df.head()

,gender,NationalITy,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,raisedhands,...,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class,at_risk,engagement_score,is_absent_frequently,parent_engaged,parent_satisfied,is_male
0,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,15,...,Yes,Good,Under-7,M,0,53,0,1,1,1
1,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,20,...,Yes,Good,Under-7,M,0,68,0,1,1,1
2,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,10,...,No,Bad,Above-7,L,1,47,1,0,0,1
3,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,30,...,No,Bad,Above-7,L,1,95,1,0,0,1
4,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,40,...,No,Bad,Above-7,M,0,152,1,0,0,1


In [6]:
le_stage = LabelEncoder()
df['StageID_encoded'] = le_stage.fit_transform(df['StageID'])

le_grade = LabelEncoder()
df['GradeID_encoded'] = le_grade.fit_transform(df['GradeID'])

le_section = LabelEncoder()
df['SectionID_encoded'] = le_section.fit_transform(df['SectionID'])

In [7]:
df_encoded = pd.get_dummies(df, columns=['NationalITy', 'PlaceofBirth', 'Topic', 'Semester', 'Relation'])
df_encoded.shape

(480, 65)

In [8]:
columns_to_drop = ['Class', 'gender', 'StudentAbsenceDays', 'ParentAnsweringSurvey', 'ParentschoolSatisfaction', 'StageID', 'GradeID', 'SectionID']
X = df_encoded.drop(columns_to_drop + ['at_risk'], axis=1)
y = df_encoded['at_risk']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (480, 56)
Target shape: (480,)


In [9]:
X.columns.tolist()

['raisedhands',
 'VisITedResources',
 'AnnouncementsView',
 'Discussion',
 'engagement_score',
 'is_absent_frequently',
 'parent_engaged',
 'parent_satisfied',
 'is_male',
 'StageID_encoded',
 'GradeID_encoded',
 'SectionID_encoded',
 'NationalITy_Egypt',
 'NationalITy_Iran',
 'NationalITy_Iraq',
 'NationalITy_Jordan',
 'NationalITy_KW',
 'NationalITy_Lybia',
 'NationalITy_Morocco',
 'NationalITy_Palestine',
 'NationalITy_SaudiArabia',
 'NationalITy_Syria',
 'NationalITy_Tunis',
 'NationalITy_USA',
 'NationalITy_lebanon',
 'NationalITy_venzuela',
 'PlaceofBirth_Egypt',
 'PlaceofBirth_Iran',
 'PlaceofBirth_Iraq',
 'PlaceofBirth_Jordan',
 'PlaceofBirth_KuwaIT',
 'PlaceofBirth_Lybia',
 'PlaceofBirth_Morocco',
 'PlaceofBirth_Palestine',
 'PlaceofBirth_SaudiArabia',
 'PlaceofBirth_Syria',
 'PlaceofBirth_Tunis',
 'PlaceofBirth_USA',
 'PlaceofBirth_lebanon',
 'PlaceofBirth_venzuela',
 'Topic_Arabic',
 'Topic_Biology',
 'Topic_Chemistry',
 'Topic_English',
 'Topic_French',
 'Topic_Geology',
 '

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("Training target distribution:")
print(y_train.value_counts())

Training set size: (384, 56)
Test set size: (96, 56)
Training target distribution:
at_risk
0    282
1    102
Name: count, dtype: int64


In [11]:
numeric_features = ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion', 'engagement_score']

scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

In [12]:
X_train.head()

,raisedhands,VisITedResources,AnnouncementsView,Discussion,engagement_score,is_absent_frequently,parent_engaged,parent_satisfied,is_male,StageID_encoded,...,Topic_History,Topic_IT,Topic_Math,Topic_Quran,Topic_Science,Topic_Spanish,Semester_F,Semester_S,Relation_Father,Relation_Mum
85,-1.511134,-1.650744,-1.379757,-1.108097,-1.795896,1,0,0,1,2,...,False,True,False,False,False,False,True,False,True,False
181,0.113544,0.233017,-0.928136,-0.350642,-0.250015,1,0,0,1,2,...,False,False,False,False,False,False,False,True,True,False
132,-1.544291,-1.559594,-1.041041,-1.432721,-1.774425,1,0,0,1,2,...,False,True,False,False,False,False,False,True,True,False
7,0.113544,-1.346911,-0.852866,-0.747404,-0.904867,0,1,1,1,1,...,False,False,True,False,False,False,True,False,True,False
239,1.506126,0.779916,1.894495,1.200337,1.660866,0,1,1,1,1,...,False,False,False,False,True,False,False,True,False,True


In [13]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Original training set:", X_train.shape)
print("Balanced training set:", X_train_balanced.shape)
print("Balanced target distribution:")
print(y_train_balanced.value_counts())

Original training set: (384, 56)
Balanced training set: (564, 56)
Balanced target distribution:
at_risk
1    282
0    282
Name: count, dtype: int64


In [14]:
import pickle

with open('data/processed/X_train.pkl', 'wb') as f:
    pickle.dump(X_train_balanced, f)

with open('data/processed/X_test.pkl', 'wb') as f:
    pickle.dump(X_test, f)

with open('data/processed/y_train.pkl', 'wb') as f:
    pickle.dump(y_train_balanced, f)

with open('data/processed/y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)

with open('data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Preprocessed data saved successfully!")

Preprocessed data saved successfully!
